# Export vascularmd mesh in format for COMSOL

In [1]:
# Read in data from swc file of bifurcation 116 and create vascularmd mesh for it
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pyvista as pv
from ArterialTree import ArterialTree

# ── Configuration ─────────────────────────────────────────────────────────────
data_folder_name = 'BG0014.CNG'
bifurcation_id   = 116
N = 48       # nodes around each cross-section circumference (must be multiple of 8)
d = 0.25     # longitudinal cross-section density
layer_ratio = [0.2, 0.4, 0.4]  # O-grid radius ratio [a, b, c], a+b+c = 1
num_a, num_b = 4, 4            # number of O-grid layers in parts a and b

swc_path = os.path.join('..', 'Data', data_folder_name, f'bifurcation_{bifurcation_id}.swc')
out_dir  = os.path.join('..', 'Output')
os.makedirs(out_dir, exist_ok=True)

# Build the vascularmd (ArterialTree) model straight from the SWC centerline data
tree = ArterialTree("patient", "Aneurisk", swc_path)
tree.model_network()                               # parametric modeling of the network
tree.compute_cross_sections(N, d, parallel=False)   # mesh cross sections
tree.mesh_surface()                                 # required internally by tree.check_mesh()
volume_mesh = tree.mesh_volume(layer_ratio, num_a, num_b)  # O-grid hex volume mesh (pv.UnstructuredGrid)

print(f"bifurcation_{bifurcation_id}: {volume_mesh.n_points} points, {volume_mesh.n_cells} cells")

volume_vtk_path = os.path.join(out_dir, f'bifurcation_{bifurcation_id}_volume_mesh.vtk')
volume_mesh.save(volume_vtk_path)
print(f"Saved volume mesh -> {os.path.abspath(volume_vtk_path)}")


Loading swc file...
Modeling the network...
Meshing bifurcations.
Meshing edges.
Meshing surface...
Meshing volume...
bifurcation_116: 93764 points, 89232 cells
Saved volume mesh -> /home/xbfl0349/arterial_network/Output/bifurcation_116_volume_mesh.vtk


In [2]:
# ── Mesh quality check ────────────────────────────────────────────────────────
# check_mesh rebuilds the volume mesh per vessel segment and per bifurcation
# and flags any whose minimum scaled Jacobian falls at/below `thres` (<= 0
# means degenerate/inverted hexahedra) -- elements COMSOL would reject or
# mis-solve. Mirrors the Editor's "Check" button (Editor.py:2946).
thres = 0
color_field, failed_edges, failed_bifs = tree.check_mesh(thres=thres)

print(f"bifurcation_{bifurcation_id} mesh quality check (scaled Jacobian <= {thres}):")
print(f"  {len(failed_edges)} vessel(s) and {len(failed_bifs)} furcation(s) failed the test.")
if failed_edges:
    print(f"  failed edges: {failed_edges}")
if failed_bifs:
    print(f"  failed bifurcations: {failed_bifs}")


Checking bif 5
Meshing volume...
0.40143301153686783
Checking edg (1, 2)
Meshing volume...
0.7051137534082746
Checking edg (6, 3)
Meshing volume...
0.6803902327803951
Checking edg (7, 4)
Meshing volume...
0.7061049405758376
bifurcation_116 mesh quality check (scaled Jacobian <= 0):
  0 vessel(s) and 0 furcation(s) failed the test.


In [3]:
# ── Convert the volume mesh to Nastran (.nas) format for COMSOL import ────────
# Note: convert the `volume_mesh` object captured right after tree.mesh_volume()
# above, not tree.get_volume_mesh() -- check_mesh() re-runs tree.mesh_volume()
# internally per segment/bifurcation as a side effect, which overwrites the
# tree's own stored volume mesh with just the last-checked fragment.
nas_path = os.path.join(out_dir, f'bifurcation_{bifurcation_id}_volume_mesh.nas')
pv.save_meshio(nas_path, volume_mesh)
print(f"Saved Nastran mesh -> {os.path.abspath(nas_path)}")


Saved Nastran mesh -> /home/xbfl0349/arterial_network/Output/bifurcation_116_volume_mesh.nas
